### Reverse Mode

Rather than propagating derivatives from inputs to outputs, we propagate them from outputs back to inputs.
Starts with a forward pass to compute the function value and store intermediate results.
Then, a backward pass computes gradients using the chain rule.

$$
\frac{df}{dv_i} = \sum_j \frac{\partial f}{\partial v_j} \cdot \frac{\partial v_j}{\partial v_i}
$$

where $v_j$ are intermediate variables that are evaluated from $v_i$

**Use Case:** Many inputs, few outputs because a single pass can compute the derivative of the output w.r.t. all inputs.

In neural networks for example, back propagation is a special case of reverse mode

In [1]:
import math

class RValue:
    
    _prev: list[tuple['RValue', float]]
    
    def __init__(self, val) -> None:
        self._prev = []
        self.val = val
        self.grad = 0.
    
    def backward(self):
        ordered_nodes: list[RValue] = []
        visited = set()
        
        def build_graph(node: RValue):
            if node in visited: return
            node.grad = 0.0
            visited.add(node)
            for parent, _ in node._prev:
                build_graph(parent)
            ordered_nodes.append(node)
        build_graph(self)
        ordered_nodes.reverse()
        
        self.grad = 1.0
        for node in ordered_nodes:
            for parent, parent_grad in node._prev:
                parent.grad += node.grad * parent_grad
    
    def __add__(self, other) -> 'RValue':
        if not isinstance(other, type(self)): other = RValue(other)
        out = RValue(self.val + other.val)
        out._prev = [(self, 1.0), (other, 1.0)]
        return out
    
    def __radd__(self, other) -> 'RValue':
        return self + other
    
    def __sub__(self, other) -> 'RValue':
        if not isinstance(other, type(self)): other = RValue(other)
        out = RValue(self.val - other.val)
        out._prev = [(self, 1.0), (other, -1.0)]
        return out
    
    def __rsub__(self, other) -> 'RValue':
        if not isinstance(other, type(self)): other = RValue(other)
        return other - self
    
    def __mul__(self, other) -> 'RValue':
        if not isinstance(other, type(self)): other = RValue(other)
        out = RValue(self.val * other.val)
        out._prev = [(self, other.val), (other, self.val)]
        return out
    
    def __rmul__(self, other) -> 'RValue':
        return self * other
    
    def __truediv__(self, other) -> 'RValue':
        if not isinstance(other, type(self)): other = RValue(other)
        out = RValue(self.val / other.val)
        out._prev = [(self, 1 / other.val), (other, -self.val / (other.val ** 2))]
        return out
    
    def __rtruediv__(self, other) -> 'RValue':
        if not isinstance(other, type(self)): other = RValue(other)
        return other / self
    
    def __pow__(self, other) -> 'RValue':
        if not isinstance(other, type(self)): other = RValue(other)
        out = RValue(self.val ** other.val)
        out._prev = [(self, other.val * self.val ** (other.val - 1)),
                     (other, self.val ** other.val * math.log(self.val))]
        return out
    
    def __rpow__(self, other) -> 'RValue':
        if not isinstance(other, type(self)): other = RValue(other)
        return other ** self
    
    def __repr__(self) -> str:
        return f'RValue({self.val}, {self.grad})'

def Output(out: RValue) -> RValue:
    out.backward()
    return out

In [2]:
x = RValue(1)
y = RValue(1)

z = Output((2*x+y)/(5*x-y))

print(x,y,z)

RValue(1, -0.4375) RValue(1, 0.4375) RValue(0.75, 1.0)


Initially, I had each RValue store a list of its "next" nodes, but this made the backward pass more complicated. Instead, each RValue now stores its "previous" nodes along with the local gradients needed for backpropagation.

My next implementation of the `backward` method used an iterative approach
```python
    def backward(self):
        self.grad = 1.0
        stack: list[RValue] = [self]
        while stack:
            node = stack.pop()
            for parent, parent_grad in node._prev:
                parent.grad += node.grad * parent_grad
                stack.append(parent)
```
but this led to incorrect gradient calculations due to multiple visits to the same node.
For example:
- input x,
- intermediate y = x * 1.0,
- output z = y + y

When backpropagating from z, y would be processed twice, leading to double counting its contribution to the gradient of x.

To fix this, I added a `visited` set to ensure each node is processed only once during backpropagation.
```python
    def backward(self):
        self.grad = 1.0
        stack: list[RValue] = [self]
        visited = set()
        while stack:
            node = stack.pop()
            if node in visited:
                continue
            visited.add(node)
            for parent, parent_grad in node._prev:
                parent.grad += node.grad * parent_grad
                stack.append(parent)
```
but this still had an issue. For example:
- input x,
- intermediate y = x * 1.0,
- intermediate z = y * 1.0,
- output out = z + y

When backpropagating from out, y would be processed once and marked as visited, so when z tried to propagate its gradient back to y, it would skip it, leading to an incorrect gradient for x.

To address this, I had to separate the ordering of nodes from the actual backpropagation by first performing a topological sort of the computation graph.
```python
    def backward(self):
        # Topological sort
        ordered_nodes: list[RValue] = []
        visited = set()
        def build_graph(node: RValue):
            if node in visited:
                return
            visited.add(node)
            node.grad = 0.0
            for parent, _ in node._prev:
                build_graph(parent)
            ordered_nodes.append(node)
        
        build_graph(self)
        ordered_nodes.reverse()
        
        self.grad = 1.0
        for node in ordered_nodes:
            for parent, parent_grad in node._prev:
                parent.grad += node.grad * parent_grad
```
This ensures that each node's gradient is computed only after all nodes that depend on it have been processed.

This approach assumes the computation graph has no cycles. In other words, it must be a Directed Acyclic Graph (DAG). Makes me wonder how recurrent neural networks handle this...